# Crypto Lab 05B — Keyed Hashing
## MACs, PRFs, HMAC, CMAC, Poly1305, and Timing-Safe Verification

**Course:** CYBR 3570 — Applied Cryptography  
**Toolkit Version:** v0.6b  

### Today's Big Question

> If anyone can compute a hash, how do we prove that a message came from someone who knows a secret key?

Last time, we studied unkeyed hash functions such as SHA-256, SHA-3, and BLAKE2. This notebook extends that discussion to **keyed hashing**.

A keyed hash can authenticate a message. That means it helps prove both that the message was not modified and that it was produced by someone who knows the shared secret key.

## Learning Objectives

By the end of this lab, you should be able to:

- explain the difference between a cryptographic hash and a MAC,
- compute and verify HMAC tags using Python,
- explain why `Hash(key || message)` is not a safe general-purpose MAC construction,
- demonstrate how replay attacks work against valid MACs,
- explain why constant-time comparison matters,
- use CMAC and Poly1305 at a high level,
- add keyed hashing utilities to your cryptographic toolkit.

## Section 1 — Setup
Run the following cell first.

In [1]:
import hashlib
import hmac
import secrets
import time

try:
    from cryptography.hazmat.primitives.cmac import CMAC
    from cryptography.hazmat.primitives.ciphers import algorithms
    from cryptography.hazmat.primitives.poly1305 import Poly1305
    CRYPTOGRAPHY_AVAILABLE = True
except Exception as exc:
    CRYPTOGRAPHY_AVAILABLE = False
    print("cryptography package not available:", exc)

## Section 2 — Concept Check
Answer these in Markdown.

1. What security property does a MAC provide that a normal hash does not?
2. Why does a MAC require a secret key?
3. Why is a MAC not the same thing as encryption?
4. What is a replay attack?
5. Why is a PRF considered stronger than a MAC?

1. A MAC provides authentication and integrity, meaning it helps verify that the message came from someone with the secret key and wasn't changed.
2. A MAC needs a secret key so only people who know the key can create a valid authentication tag.
3. A MAC checks if a message is authentic, while encryption hides the message's contents.
4. A replay attack happens when someone captures a valid message and sends it again to trick the system.
5. A PRF is stronger because it produces outputs that look random for different inputs, while a MAC mainly guarantees that attackers cannot forge valid tags.


## Section 3 — A Plain Hash Is Public
Anyone can compute `SHA256(message)`. That is useful for integrity checks, but it does not prove who created the hash.

In [2]:
def sha256_digest(message: bytes) -> str:
    """Return the SHA-256 digest of a message as a hex string."""
    return hashlib.sha256(message).hexdigest()

message = b"Transfer $100 to account 12345"
print(sha256_digest(message))

09d11180d38974a9eea9725e077f43dd4de63173bf3c33b7ba93803f24ace456


### Reflection
Suppose Alice sends Bob this message and digest:

```text
Transfer $100 to account 12345
SHA256(message)
```

Why does this not prove that Alice created the message?

Anyone can calculate a SHA-256 hash, so the digest doesn't prove Alice created the message. An attacker could change the message and calculate a new hash. A normal hash doesn't use a secret key to verify who sent it. Copy


## Section 4 — HMAC: A Standard Keyed Hash Construction
Python's `hmac` module implements HMAC. HMAC combines a secret key, a message, and a hash function.

In [3]:
def hmac_sha256(key: bytes, message: bytes) -> bytes:
    """Compute HMAC-SHA-256(key, message)."""
    return hmac.new(key, message, hashlib.sha256).digest()

key = secrets.token_bytes(32)
message = b"Transfer $100 to account 12345"
tag = hmac_sha256(key, message)

print("key:", key.hex())
print("tag:", tag.hex())

key: ff9cf5097f16bc53df3c273e048c69f7051bd4d6b422d255d6744c8915d9e7f7
tag: d25338ecb3de5b64e177d5c4dff75727e604471042f775501b910fb3d4a81b05


## Exercise 1 — Verify an HMAC Tag
Complete the function below.

Requirements:

- recompute the HMAC tag for the received message,
- compare the expected and received tags using `hmac.compare_digest`,
- return `True` only if the tag is valid.

In [5]:

def verify_hmac_sha256(key: bytes, message: bytes, received_tag: bytes) -> bool:
    """Verify an HMAC-SHA256 tag."""
    expected_tag = hmac.new(key, message, hashlib.sha256).digest()
    return hmac.compare_digest(expected_tag, received_tag)


valid = verify_hmac_sha256(key, message, tag)
modified = verify_hmac_sha256(
    key, b"Transfer $900 to account 12345", tag
)

print("valid tag verifies:", valid)
print("modified message verifies:", modified)


valid tag verifies: True
modified message verifies: False


## Section 5 — Why Not Just Hash `key || message`?
A tempting but dangerous construction is:

```python
SHA256(key + message)
```

This is called a **secret-prefix MAC**. It looks plausible, but it is not a safe general-purpose design. With Merkle–Damgård hashes such as SHA-256, it can be vulnerable to length-extension attacks.

In [6]:
def insecure_secret_prefix_mac(key: bytes, message: bytes) -> bytes:
    """Educational anti-pattern: do not use this as a MAC."""
    return hashlib.sha256(key + message).digest()

bad_tag = insecure_secret_prefix_mac(key, message)
print(bad_tag.hex())

15c0bbe5f6f44995a80750a679bee56f149360b3fc8a0c516322c1f2240f0a42


### Exercise 2 — Explain the Anti-Pattern
In your own words, explain why `SHA256(key || message)` is not a good replacement for HMAC.

Your answer should mention at least one of the following: length-extension attacks, accidental design mistakes, ambiguity about key/message boundaries, or the availability of standard constructions.

Using SHA256(key + message) instead of HMAC is unsafe because it can be vulnerable to length-extension attacks. An attacker might be able to create a valid hash for a modified message without knowing the secret key. HMAC is better because it was specifically designed for secure message authentication.


## Section 6 — Replay Attacks
A valid MAC proves that a message/tag pair was produced by someone who knows the key. It does **not** prove that the message is fresh.

In [7]:
shared_key = secrets.token_bytes(32)

payment = b"PAY DEREK 100"
payment_tag = hmac_sha256(shared_key, payment)

print("Original verifies:", verify_hmac_sha256(shared_key, payment, payment_tag))
print("Replay verifies:", verify_hmac_sha256(shared_key, payment, payment_tag))

Original verifies: True
Replay verifies: True


### Exercise 3 — Add a Message Number
Modify the message format so that the MAC authenticates both the message number and the message body.

Example format:

```text
000001|PAY DEREK 100
000002|PAY DEREK 100
```

Then explain why a receiver must track which message numbers have already been accepted.

The receiver needs to keep track of message numbers so attackers can't reuse old messages. Even if an old message has a valid MAC, the receiver should reject it if the message number was already used. This helps prevent replay attacks. Copy


In [9]:

def encode_numbered_message(number: int, body: bytes) -> bytes:
    """Encode a message number and message body."""
    if number < 0 or number > 999999:
        raise ValueError("Message number must be between 0 and 999999")
    return f"{number:06d}|".encode() + body


numbered = encode_numbered_message(1, b"PAY DEREK 100")
numbered_tag = hmac_sha256(shared_key, numbered)

print(numbered)
print(numbered_tag.hex())


b'000001|PAY DEREK 100'
4f33dc6a53f26e9eded3aa51abcdbf2b0e9244c0f74e90eb32480c4b62fd159f


## Section 7 — Constant-Time Comparison
A common implementation mistake is to compare tags byte by byte and return as soon as the first mismatch is found. That creates a timing leak.

In [10]:
def insecure_compare(x: bytes, y: bytes) -> bool:
    """Educational anti-pattern: variable-time comparison."""
    if len(x) != len(y):
        return False
    for i in range(len(x)):
        if x[i] != y[i]:
            return False
    return True

correct = hmac_sha256(shared_key, b"important message")
wrong_first = bytes([correct[0] ^ 1]) + correct[1:]
wrong_last = correct[:-1] + bytes([correct[-1] ^ 1])

print(insecure_compare(correct, correct))
print(insecure_compare(correct, wrong_first))
print(insecure_compare(correct, wrong_last))

True
False
False


### Exercise 4 — Measure Timing Differences
Complete the measurement function below and compare:

- correct tag vs. correct tag,
- correct tag vs. tag wrong in the first byte,
- correct tag vs. tag wrong in the last byte,
- the same cases using `hmac.compare_digest`.

In [12]:

import time
import hmac

def insecure_compare(a: bytes, b: bytes) -> bool:
    if len(a) != len(b):
        return False
    for x, y in zip(a, b):
        if x != y:
            return False
    return True

def time_comparison(compare_func, a: bytes, b: bytes, trials: int = 100_000) -> float:
    """Return elapsed time for repeated comparison calls."""
    start = time.perf_counter()
    for _ in range(trials):
        compare_func(a, b)
    end = time.perf_counter()
    return end - start

correct = hmac.new(b"secret_key", b"Hello", "sha256").digest()

wrong_first = bytes([correct[0] ^ 1]) + correct[1:]
wrong_last = correct[:-1] + bytes([correct[-1] ^ 1])

print("Insecure comparison:")
print("Correct vs correct:", time_comparison(insecure_compare, correct, correct))
print("Wrong first byte:", time_comparison(insecure_compare, correct, wrong_first))
print("Wrong last byte:", time_comparison(insecure_compare, correct, wrong_last))

print("\nSecure comparison:")
print("Correct vs correct:", time_comparison(hmac.compare_digest, correct, correct))
print("Wrong first byte:", time_comparison(hmac.compare_digest, correct, wrong_first))
print("Wrong last byte:", time_comparison(hmac.compare_digest, correct, wrong_last))


Insecure comparison:
Correct vs correct: 0.17461240012198687
Wrong first byte: 0.0487937000580132
Wrong last byte: 0.1911321000661701

Secure comparison:
Correct vs correct: 0.01007630000822246
Wrong first byte: 0.01055390015244484
Wrong last byte: 0.010579599998891354


## Section 8 — CMAC: A Block-Cipher-Based MAC
HMAC builds a MAC from a hash function. CMAC builds a MAC from a block cipher such as AES.

In [15]:
if CRYPTOGRAPHY_AVAILABLE:
    cmac_key = secrets.token_bytes(16)  # AES-128 key
    c = CMAC(algorithms.AES(cmac_key))
    c.update(b"message authenticated with AES-CMAC")
    cmac_tag = c.finalize()
    print("CMAC tag:", cmac_tag.hex())
else:
    print("Install cryptography to run the CMAC example.")

CMAC tag: 0f27154c195b4482291e02addce169c3


### Exercise 5 — Compare HMAC and CMAC
Answer in Markdown:

1. What primitive does HMAC build from?
2. What primitive does CMAC build from?
3. Why might a protocol choose CMAC instead of HMAC?
4. Why should you avoid implementing CBC-MAC yourself?

1. What primitive does HMAC build from?
HMAC uses a cryptographic hash function, such as SHA-256.
2. What primitive does CMAC build from?
CMAC uses a block cipher, such as AES.
3. Why might a protocol choose CMAC instead of HMAC?
CMAC might be chosen if the system already uses AES because it can reuse the same type of cryptographic hardware.
4. Why should you avoid implementing CBC-MAC yourself?
It's easy to make mistakes that create security problems. Using a trusted library is safer.

## Section 9 — Poly1305: A Fast Dedicated MAC
Poly1305 is a fast MAC often paired with ChaCha20 in modern authenticated encryption.

In [16]:
if CRYPTOGRAPHY_AVAILABLE:
    poly_key = secrets.token_bytes(32)
    poly_tag = Poly1305.generate_tag(poly_key, b"message authenticated with Poly1305")
    print("Poly1305 tag:", poly_tag.hex())
else:
    print("Install cryptography to run the Poly1305 example.")

Poly1305 tag: e5cc7dcb3b37ff6dfccb0f67455a9d4f


## Section 10 — Toolkit Integration
Add a new module to your toolkit:

```text
crypto_toolkit/
    hashes/
        keyed.py
```

Your module should include:

- `hmac_sha256(key: bytes, message: bytes) -> bytes`
- `verify_hmac_sha256(key: bytes, message: bytes, tag: bytes) -> bool`
- `sha256_digest(message: bytes) -> bytes`
- clear docstrings warning against naive keyed-hash constructions

Optional challenge: add `cmac_aes()` and `poly1305_tag()`.

In [17]:
# Starter code for crypto_toolkit/hashes/keyed.py

MODULE_STARTER = """
"""
Module: crypto_toolkit.hashes.keyed

Educational keyed hashing utilities for CYBR 3570.

WARNING
-------
These wrappers are for learning. Production applications should rely on
well-reviewed libraries and protocol-level constructions.
"""

import hashlib
import hmac


def sha256_digest(message: bytes) -> bytes:
    """Return SHA-256(message)."""
    return hashlib.sha256(message).digest()


def hmac_sha256(key: bytes, message: bytes) -> bytes:
    """Return HMAC-SHA-256(key, message)."""
    return hmac.new(key, message, hashlib.sha256).digest()


def verify_hmac_sha256(key: bytes, message: bytes, tag: bytes) -> bool:
    """Verify HMAC-SHA-256 using constant-time comparison."""
    expected = hmac_sha256(key, message)
    return hmac.compare_digest(expected, tag)
"""

print(MODULE_STARTER)

SyntaxError: invalid syntax (3416369380.py, line 7)

## Section 11 — Security Engineering Reflection
Answer these in Markdown.

1. Why is encryption without authentication dangerous?
2. Why does a valid MAC not necessarily mean a message is fresh?
3. Why is constant-time comparison part of cryptographic engineering rather than just performance engineering?
4. A developer says, “I used SHA-256, so my MAC is secure.” What questions would you ask?
5. What is the connection between this week's material and authenticated encryption?

1. Why is encryption without authentication dangerous?
Encryption hides the message, but it doesn't stop someone from changing it. Authentication helps make sure the message hasn't been modified.
2. Why does a valid MAC not necessarily mean a message is fresh?
A valid MAC proves the message came from someone with the secret key, but an attacker could copy and resend an old message.
3. Why is constant-time comparison part of cryptographic engineering rather than just performance engineering?
Constant-time comparison helps prevent attackers from using timing differences to guess secret information. It's about security, not just speed.
4. A developer says, "I used SHA-256, so my MAC is secure." What questions would you ask?
I would ask how they used SHA-256, whether they used HMAC, how they protect the secret key, and whether their system prevents replay attacks.
5. What is the connection between this week's material and authenticated encryption?
This week's material shows why authentication is important. Authenticated encryption combines encryption and authentication to keep messages private and detect changes.

## Submission Checklist

- [ y] I answered all concept questions.
- [ y] I completed `verify_hmac_sha256()`.
- [ y] I completed the replay/message-number exercise.
- [ y] I ran the timing comparison experiment.
- [ y] I added `crypto_toolkit/hashes/keyed.py` to my toolkit.
- [ y] I committed and pushed my changes.